# Loan Default Prediction
### Original example by Danny Barren: *ML for Business Insights, Examples*

**The business question:** *Which loan applications are most likely to default, and which ones are worth sending to a manual review before we approve them?*

A lender loses money twice: once when a risky loan defaults, and again when a good customer gets turned away or delayed. This notebook builds a simple, explainable **logistic regression** model that:

1. Scores every application with a **probability of default**
2. Compares that score against a "do nothing smart" baseline
3. Picks a **review threshold** using real money: the loss you avoid versus the cost of each review
4. Explains, in plain English, **which applicant traits push risk up or down**

**How to use it:** Edit only the **CONFIG** cell, then run top to bottom (*Runtime → Run all* in Colab).
If `CSV_PATH = None`, the notebook creates a **synthetic** demo portfolio so you can see every step working first.

> This is an inspiration example, not a credit policy. Real lending decisions must follow fair-lending law and your compliance team's rules (for example, never use protected attributes such as race, religion, sex, or age as inputs).

In [ ]:
# ============================================================
# IMPORTS: everything we need, loaded once (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")   # keep the output readable for business users

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_validate, cross_val_predict, GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================
CSV_PATH = None                 # e.g. "/content/loans.csv"; None = synthetic demo data

ID_COLUMN = "loan_id"           # a unique ID per application (kept out of the model)
TARGET_COLUMN = "defaulted"     # the yes/no outcome we want to predict
POSITIVE_VALUE = 1              # the value that means "defaulted" (1, "Yes", "default", ...)

# Inputs known AT APPLICATION TIME. Never include anything learned afterwards
# (e.g. "days_past_due", "collections_flag", "charge_off_amount"); that is leakage.
NUMERIC_COLUMNS = ["credit_score", "annual_income", "loan_amount", "debt_to_income",
                   "months_employed", "late_payments_24m", "loan_term_months"]
CATEGORICAL_COLUMNS = ["loan_purpose", "home_ownership"]

DROP_COLUMNS = []               # anything else to ignore (notes, free text, etc.)

# --- Business numbers (edit to match your lender) ----------------
VALUE_IF_AVOIDED = 4_000        # average loss avoided when a would-be default is caught and declined/restructured
REVIEW_COST = 150               # cost of one manual underwriting review (staff time, docs)
REVIEW_CATCH_RATE = 0.50        # share of real defaulters that the review actually stops
FALSE_ALARM_COST = 400          # profit lost when a GOOD applicant is flagged (delays, walk-aways)

MAIN_METRIC = "average_precision"   # PR-AUC: best when defaults are rare (<20%); or "roc_auc"
DATA_SIZE_MODE = "auto"         # "auto", "small" (<1,000 rows), or "large"
TEST_SIZE = 0.25                # share of rows held back for the final, honest exam
RANDOM_STATE = 42               # makes results repeatable

In [ ]:
# ============================================================
# LOAD DATA (or build a synthetic demo portfolio)
# ============================================================
def make_demo_loans(n=800, seed=RANDOM_STATE):
    """Synthetic loan book with a realistic story:
    low credit score, high debt-to-income, past late payments, long terms
    and big loans relative to income all raise default risk."""
    rng = np.random.default_rng(seed)
    credit = np.clip(rng.normal(680, 60, n), 450, 850).round()
    income = np.round(rng.lognormal(np.log(62_000), 0.45, n), -2)
    loan = np.round(rng.lognormal(np.log(14_000), 0.6, n), -2)
    dti = np.clip(rng.beta(2.2, 5, n) * 0.8, 0.02, 0.75).round(3)
    employed = np.clip(rng.exponential(60, n), 0, 400).round()
    late = rng.poisson(0.6, n)
    term = rng.choice([36, 60], n, p=[0.65, 0.35])
    purpose = rng.choice(["debt_consolidation", "home_improvement", "car", "small_business", "other"],
                         n, p=[0.40, 0.20, 0.15, 0.10, 0.15])
    home = rng.choice(["rent", "mortgage", "own"], n, p=[0.45, 0.40, 0.15])
    loan_to_income = loan / income
    logit = (-2.9 - 0.022 * (credit - 680) + 6.0 * (dti - 0.25) + 0.6 * late
             + 0.6 * (term == 60) + 0.9 * (purpose == "small_business")
             + 0.4 * (home == "rent") - 0.006 * (employed - 60) + 3.0 * (loan_to_income - 0.25))
    default = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame({"loan_id": np.arange(100_001, 100_001 + n), "credit_score": credit,
                       "annual_income": income, "loan_amount": loan, "debt_to_income": dti,
                       "months_employed": employed, "late_payments_24m": late,
                       "loan_term_months": term, "loan_purpose": purpose,
                       "home_ownership": home, "defaulted": default})
    # Real data has gaps; add a few so the pipeline's imputers get exercised
    for col in ["months_employed", "annual_income"]:
        df.loc[rng.random(n) < 0.03, col] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_loans()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")

print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK: catch typos before they cause confusing errors
# ============================================================
needed = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in needed if c not in df.columns]
assert not missing, f"These CONFIG columns are not in your file: {missing}"

# Turn the target into a clean 0/1 column: 1 = defaulted, 0 = repaid
y_all = (df[TARGET_COLUMN] == POSITIVE_VALUE).astype(int)
assert y_all.nunique() == 2, "The target needs both defaults and non-defaults. Check POSITIVE_VALUE."

FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X_all = df[FEATURES].copy()
for c in NUMERIC_COLUMNS:                     # force numbers (e.g. "1,200" stored as text)
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce").astype(float)
for c in CATEGORICAL_COLUMNS:
    X_all[c] = X_all[c].astype("string").astype(object)

print(f"Default rate: {y_all.mean():.1%}  ({y_all.sum():,} defaults out of {len(y_all):,})")
print("Config looks good.")

## 1. Light EDA: what does the portfolio look like?
We only look at what matters for a first model: missing values, the overall default rate, and whether the obvious risk factors actually separate good loans from bad ones.

In [ ]:
# Missing values and basic ranges: big gaps or odd ranges are worth a conversation with the data owner
quality = pd.DataFrame({"missing_%": X_all.isna().mean() * 100,
                        "unique_values": X_all.nunique()})
display(quality.round(1))
display(X_all[NUMERIC_COLUMNS].describe().T[["mean", "min", "50%", "max"]])

In [ ]:
# Default rate by each category, and by credit-score band (if that column exists)
n_plots = len(CATEGORICAL_COLUMNS) + 1
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 3.8))
for ax, col in zip(axes, CATEGORICAL_COLUMNS):
    rates = y_all.groupby(X_all[col]).mean().sort_values()
    rates.plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"Default rate by {col}"); ax.set_xlabel("default rate")
first_num = NUMERIC_COLUMNS[0]
bands = pd.qcut(X_all[first_num], 5, duplicates="drop")
y_all.groupby(bands).mean().plot.bar(ax=axes[-1], color="indianred")
axes[-1].set_title(f"Default rate by {first_num} (5 bands)"); axes[-1].set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# LEAKAGE SNIFF TEST: if a single column predicts default almost perfectly on its own,
# it was probably recorded AFTER the loan went bad. Investigate before trusting the model.
rows = []
for c in NUMERIC_COLUMNS:
    s = X_all[c].fillna(X_all[c].median())
    auc = roc_auc_score(y_all, s)
    rows.append((c, max(auc, 1 - auc)))
sniff = pd.DataFrame(rows, columns=["column", "single_column_AUC"]).sort_values("single_column_AUC", ascending=False)
display(sniff)
suspects = sniff.loc[sniff.single_column_AUC > 0.90, "column"].tolist()
print("Possible leakage, check these:" if suspects else "No obvious leakage (no single column above 0.90 AUC).", suspects or "")

## 2. Split the data and choose a validation plan
We lock away a **test set** now and do not touch it until the very end, so the final score is an honest preview of real life.
With small data we repeat cross-validation several times so one lucky split can't fool us.

In [ ]:
n_rows = len(df)
size_mode = DATA_SIZE_MODE if DATA_SIZE_MODE != "auto" else ("small" if n_rows < 1_000 else "large")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=RANDOM_STATE)

if size_mode == "small":
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)  # 15 fits per setting
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)        # 5 fits is plenty

print(f"Data size mode: {size_mode}  |  train {len(X_train):,} rows, test {len(X_test):,} rows")
print(f"Default rate train {y_train.mean():.1%} vs test {y_test.mean():.1%} (should be similar)")

## 3. A leakage-safe pipeline
The **Pipeline** bundles cleaning and modelling into one object. Inside cross-validation, the imputers and scaler learn **only from the training folds**, so no information sneaks in from the rows being scored.

- Numbers: fill gaps with the median, then put everything on the same scale (needed for fair coefficients)
- Categories: fill gaps with the most common value, then one-hot encode (unseen categories are safely ignored)

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUMERIC_COLUMNS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
     CATEGORICAL_COLUMNS),
], verbose_feature_names_out=False)

logit_pipe = Pipeline([("prep", preprocess),
                       ("model", LogisticRegression(max_iter=2_000))])
logit_pipe

## 4. Baseline first: is the model actually better than guessing?
The **Dummy** model ignores every input and always predicts the average default rate. Any real model must clearly beat it. We also show training vs. validation scores: a big gap means the model is memorising instead of learning.

In [ ]:
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision"}

def cv_summary(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    return {"model": name,
            "ROC_AUC": r["test_roc_auc"].mean(), "PR_AUC": r["test_average_precision"].mean(),
            "PR_AUC_spread": r["test_average_precision"].std(),
            "train_minus_valid_PR_AUC": r["train_average_precision"].mean() - r["test_average_precision"].mean()}

baseline = DummyClassifier(strategy="prior")
results = pd.DataFrame([cv_summary("Dummy (always the average)", baseline),
                        cv_summary("Logistic regression (defaults)", logit_pipe)])
display(results)
print("Reading tip: Dummy PR-AUC equals the default rate. Anything well above it is real signal.")

## 5. Light tuning: how much should the model "trust" the data?
Logistic regression has one main dial, **C**:
- **small C** = strong regularization, a cautious and simple model (good for small data)
- **large C** = the model follows the data closely (can overfit)

We also try `class_weight="balanced"`, which tells the model to pay extra attention to the rarer defaults.
**Simplicity rule:** among all settings within one standard error of the best score, we pick the most cautious (smallest C). It usually holds up better on new applications.

In [ ]:
param_grid = {"model__C": [0.03, 0.1, 0.3, 1, 3, 10],
              "model__class_weight": [None, "balanced"]}
search = GridSearchCV(logit_pipe, param_grid, cv=cv, scoring=MAIN_METRIC, n_jobs=-1)
search.fit(X_train, y_train)

res = pd.DataFrame(search.cv_results_)
res["C"] = res["param_model__C"].astype(float)
res["class_weight"] = res["param_model__class_weight"].apply(lambda v: "balanced" if v == "balanced" else "none")
n_splits_total = cv.get_n_splits()
best = res.loc[res["rank_test_score"].idxmin()]
one_se = best["std_test_score"] / np.sqrt(n_splits_total)
close_enough = res[res["mean_test_score"] >= best["mean_test_score"] - one_se]
# simplest = smallest C first; on a tie prefer no re-weighting ("balanced" sorts after "none" when reversed)
chosen = close_enough.assign(_cw=(close_enough["class_weight"] == "balanced").astype(int)).sort_values(["C", "_cw"]).iloc[0]

display(res[["C", "class_weight", "mean_test_score", "std_test_score", "rank_test_score"]]
        .sort_values("rank_test_score").head(8))
print(f"Best score: C={best['C']}, class_weight={best['class_weight']} -> {best['mean_test_score']:.3f}")
print(f"Chosen (simplest within 1 SE): C={chosen['C']}, class_weight={chosen['class_weight']} -> {chosen['mean_test_score']:.3f}")

final_model = clone(logit_pipe).set_params(
    model__C=chosen["C"], model__class_weight=None if chosen["class_weight"] == "none" else "balanced")
final_model.fit(X_train, y_train)

## 6. The honest exam: score the locked-away test set
- **ROC-AUC**: if you pick one defaulter and one good borrower at random, how often does the model rank the defaulter as riskier? 0.5 = coin flip, 1.0 = perfect.
- **PR-AUC**: how clean the "high risk" list is. Compare it to the default rate (what random guessing gets).

In [ ]:
p_test = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, p_test)
test_ap = average_precision_score(y_test, p_test)
print(f"Test ROC-AUC: {test_auc:.3f}   (coin flip = 0.500)")
print(f"Test PR-AUC : {test_ap:.3f}   (random guessing = {y_test.mean():.3f}, the default rate)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[0]); axes[0].plot([0, 1], [0, 1], "k--", lw=1)
PrecisionRecallDisplay.from_predictions(y_test, p_test, ax=axes[1])
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label="random")
axes[0].set_title("ROC curve"); axes[1].set_title("Precision-recall curve"); plt.tight_layout(); plt.show()

## 7. Turn scores into a decision: which applications go to manual review?
A probability is not a decision. We test many cut-offs and calculate the **expected money** for each:

> value = (defaults caught × `REVIEW_CATCH_RATE` × `VALUE_IF_AVOIDED`) − (reviews × `REVIEW_COST`) − (good applicants flagged × `FALSE_ALARM_COST`)

The cut-off is chosen on **out-of-fold predictions from the training data** (never on the test set), then checked once on the test set.

In [ ]:
oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
p_oof = cross_val_predict(final_model, X_train, y_train, cv=oof_cv, method="predict_proba")[:, 1]

def value_table(y_true, p, thresholds):
    rows = []
    for t in thresholds:
        flag = p >= t
        tp = int((flag & (y_true == 1)).sum()); fp = int((flag & (y_true == 0)).sum())
        value = tp * REVIEW_CATCH_RATE * VALUE_IF_AVOIDED - flag.sum() * REVIEW_COST - fp * FALSE_ALARM_COST
        rows.append({"threshold": t, "flagged_%": flag.mean() * 100, "defaults_caught": tp,
                     "good_flagged": fp, "recall": tp / max(1, (y_true == 1).sum()),
                     "precision": tp / max(1, flag.sum()),
                     "value_per_1000_apps": value / len(y_true) * 1000})
    return pd.DataFrame(rows)

thresholds = np.round(np.arange(0.05, 0.81, 0.05), 2)
tbl = value_table(y_train.values, p_oof, thresholds)
best_t = tbl.loc[tbl["value_per_1000_apps"].idxmax(), "threshold"]
display(tbl.style.format({"flagged_%": "{:.1f}", "recall": "{:.0%}", "precision": "{:.0%}",
                          "value_per_1000_apps": "${:,.0f}"}).highlight_max("value_per_1000_apps", color="#c6efce"))

break_even = (REVIEW_COST + FALSE_ALARM_COST) / (REVIEW_CATCH_RATE * VALUE_IF_AVOIDED + FALSE_ALARM_COST)
print(f"Best review threshold (out-of-fold): {best_t:.2f}")
print(f"Rule of thumb: a review pays for itself when P(default) is above about {break_even:.2f}")

In [ ]:
# Check the chosen threshold ONCE on the test set
flag_test = p_test >= best_t
tn, fp, fn, tp = confusion_matrix(y_test, flag_test).ravel()
test_value = tp * REVIEW_CATCH_RATE * VALUE_IF_AVOIDED - flag_test.sum() * REVIEW_COST - fp * FALSE_ALARM_COST
review_all = y_test.sum() * REVIEW_CATCH_RATE * VALUE_IF_AVOIDED - len(y_test) * REVIEW_COST - (y_test == 0).sum() * FALSE_ALARM_COST
print(f"Test set at threshold {best_t:.2f}: {flag_test.sum()} of {len(y_test)} applications sent to review")
print(f"  defaults caught {tp}, defaults missed {fn}, good applicants flagged {fp}")
money = lambda v: f"-${abs(v):,.0f}" if v < 0 else f"${v:,.0f}"
print(f"  expected value per 1,000 applications: {money(test_value / len(y_test) * 1000)}")
print(f"  compare: review nobody = $0, review everybody = {money(review_all / len(y_test) * 1000)}")

## 8. What drives default risk? (plain-language interpretation)
Because numbers were standardized, each coefficient tells us what happens when that input moves by **one standard deviation** (one "typical step"). We convert it to an **odds multiplier**:
- above 1.0 → raises the odds of default
- below 1.0 → lowers the odds of default

For categories, the multiplier compares that group with the average applicant.

In [ ]:
coefs = pd.Series(final_model.named_steps["model"].coef_[0],
                  index=final_model.named_steps["prep"].get_feature_names_out())
odds = pd.DataFrame({"coefficient": coefs, "odds_multiplier": np.exp(coefs)})
odds["direction"] = np.where(odds["coefficient"] > 0, "raises risk", "lowers risk")
odds = odds.reindex(odds["coefficient"].abs().sort_values(ascending=False).index)
display(odds)

train_sd = X_train[NUMERIC_COLUMNS].std()
print("In plain English:")
for feat, row in odds.head(6).iterrows():
    change = (row["odds_multiplier"] - 1) * 100
    if feat in NUMERIC_COLUMNS:
        print(f"  • Each +{train_sd[feat]:,.2f} in {feat} (one typical step) changes the odds of default by {change:+.0f}%.")
    else:
        print(f"  • Applicants with {feat} have {change:+.0f}% odds of default versus the average applicant.")

In [ ]:
# Second opinion: PERMUTATION IMPORTANCE. Shuffle one column at a time on the test set and
# measure how much the model's ranking gets worse. Bigger drop = the model relies on it more.
perm = permutation_importance(final_model, X_test, y_test, scoring=MAIN_METRIC,
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 4), color="teal"); plt.title(f"Permutation importance (drop in {MAIN_METRIC})")
plt.tight_layout(); plt.show()

In [ ]:
# Risk tiers: an easy way to hand the model to an underwriting team
low_cut = min(break_even, best_t) / 2
tiers = pd.cut(p_test, [0, low_cut, best_t, 1.0], labels=["Low: auto-approve path",
               "Medium: standard checks", "High: manual review"], include_lowest=True)
tier_tbl = pd.DataFrame({"tier": tiers, "defaulted": y_test.values}).groupby("tier", observed=False).agg(
    applications=("defaulted", "size"), actual_default_rate=("defaulted", "mean"))
display(tier_tbl.style.format({"actual_default_rate": "{:.1%}"}))

## 9. What to do next: business recommendations
**Use it**
- Route **High** tier applications to manual review; keep **Low** tier on the fast path to protect the customer experience.
- Use the top drivers above to write **review checklists** (for example, verify income when loan size is large relative to income).
- Re-run the threshold table whenever review costs, loss rates, or team capacity change. The best cut-off moves with the economics.

**Prove it**
- Pilot on a share of new applications (e.g. 20%) and compare default and approval rates against business-as-usual for one or two quarters.

**Keep it healthy**
- Track the monthly default rate per tier. If High-tier loans stop defaulting more often than Low-tier ones, retrain.
- Have compliance review the inputs for fairness and explainability (adverse-action reasons) before production use.

**Small vs. large data**
- *Small (< 1,000 loans):* keep logistic regression, repeated CV, and smaller C values; treat drivers as directional.
- *Large (10k+ loans):* single 5-fold CV is enough; try `RandomForestClassifier` or `HistGradientBoostingClassifier` as a challenger and keep it only if PR-AUC and expected value clearly improve.

In [ ]:
# Optional: save the trained pipeline (cleaning + model in one file) for scoring new applications
import joblib
joblib.dump({"model": final_model, "threshold": float(best_t), "features": FEATURES}, "loan_default_model.joblib")
print("Saved loan_default_model.joblib. Load it later with joblib.load() and call .predict_proba(new_df[FEATURES]).")
print(f"\nSummary: test ROC-AUC {test_auc:.2f}, PR-AUC {test_ap:.2f} vs {y_test.mean():.2f} random; "
      f"review threshold {best_t:.2f}; top driver: {odds.index[0]}.")